## Team Names :
Adham hesham 

Mohamed moktar

In [ ]:
!pip install -q -U langchain langgraph langgraph-checkpoint-sqlite langchain-google-genai "gradio>=5,<6" langchain-classic

## 1 · Config & model

In [ ]:
import os, json, time, sqlite3, uuid, getpass
from contextlib import contextmanager
from datetime import datetime
from typing import Literal
from pydantic import BaseModel, Field

if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
MODEL_NAME = "google_genai:gemini-2.5-flash"   # any chat model that supports tool calling
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
MEMORY_DB = f"{OUT_DIR}/long_term_memory.db"        # semantic / episodic / procedural memory
CHECKPOINT_DB = f"{OUT_DIR}/short_term_checkpoints.db"  # per-thread conversation history
DEMO_DELAY = 0.8   # seconds each mock tool "thinks" so the ⟳ Running state is visible in the UI

from langchain.chat_models import init_chat_model
llm = init_chat_model(MODEL_NAME, temperature=0)
print("Model ready:", MODEL_NAME)

## 2 · Long-term memory store (SQLite) — survives restarts
Three kinds of long-term memory, following the LangGraph taxonomy:
* **Semantic** – facts about the user (`preferences`: likes / avoids / budget)
* **Episodic** – things that happened (`episodes`: past planning sessions)
* **Procedural** – how the agent should behave (`procedures`: rules, incl. ones the user teaches it)

In [ ]:
class MemoryStore:
    DEFAULT_RULES = [
        "Present itineraries day by day with approximate times.",
        "Always apply the user's saved preferences and say which ones you applied.",
        "Never ask again for something that is already in memory.",
    ]

    def __init__(self, path):
        self.path = path
        with self._db() as c:
            c.executescript("""
            CREATE TABLE IF NOT EXISTS preferences(
                user_id TEXT, category TEXT, value TEXT COLLATE NOCASE, updated_at TEXT,
                PRIMARY KEY(user_id, category, value));
            CREATE TABLE IF NOT EXISTS episodes(
                id INTEGER PRIMARY KEY AUTOINCREMENT, user_id TEXT, summary TEXT, created_at TEXT);
            CREATE TABLE IF NOT EXISTS procedures(
                user_id TEXT, rule TEXT, PRIMARY KEY(user_id, rule));
            """)

    @contextmanager
    def _db(self):
        con = sqlite3.connect(self.path, timeout=30)
        try:
            yield con
            con.commit()
        finally:
            con.close()

    # semantic
    def save_preference(self, user_id, category, value):
        value = value.strip()
        if not value:
            return
        with self._db() as c:
            if category == "budget":                      # budget is single-valued
                c.execute("DELETE FROM preferences WHERE user_id=? AND category='budget'", (user_id,))
            c.execute("INSERT OR REPLACE INTO preferences VALUES(?,?,?,?)",
                      (user_id, category, value, datetime.now().isoformat(timespec="seconds")))

    # episodic
    def add_episode(self, user_id, summary):
        with self._db() as c:
            c.execute("INSERT INTO episodes(user_id, summary, created_at) VALUES(?,?,?)",
                      (user_id, summary, datetime.now().strftime("%Y-%m-%d %H:%M")))

    # procedural
    def add_rule(self, user_id, rule):
        with self._db() as c:
            c.execute("INSERT OR IGNORE INTO procedures VALUES(?,?)", (user_id, rule.strip()))

    def get_all(self, user_id, n_episodes=5):
        with self._db() as c:
            prefs = c.execute("SELECT category, value FROM preferences WHERE user_id=? ORDER BY category, value",
                              (user_id,)).fetchall()
            eps = c.execute("SELECT created_at, summary FROM episodes WHERE user_id=? ORDER BY id DESC LIMIT ?",
                            (user_id, n_episodes)).fetchall()
            rules = [r[0] for r in c.execute("SELECT rule FROM procedures WHERE user_id=?", (user_id,))]
        return {"preferences": prefs, "episodes": eps, "rules": self.DEFAULT_RULES + rules}

    def reset(self, user_id):
        with self._db() as c:
            for t in ("preferences", "episodes", "procedures"):
                c.execute(f"DELETE FROM {t} WHERE user_id=?", (user_id,))

memory = MemoryStore(MEMORY_DB)

## 3 · Mock tools (5) — predefined fake JSON, no real APIs

In [ ]:
from langchain_core.tools import tool

def _pause():
    time.sleep(DEMO_DELAY)

HOTELS = [
    {"name": "Hostel Santa Maria", "price_per_night_eur": 70, "stars": 2, "area": "Centro"},
    {"name": "Hotel Davanzati", "price_per_night_eur": 145, "stars": 3, "area": "Duomo"},
    {"name": "Grand Palazzo Firenze", "price_per_night_eur": 320, "stars": 5, "area": "Arno riverside"},
]
ATTRACTIONS = [
    {"name": "Uffizi Gallery", "type": "museum", "start_time": "09:00", "price_eur": 25},
    {"name": "Galleria dell'Accademia (David)", "type": "museum", "start_time": "10:00", "price_eur": 16},
    {"name": "Palazzo Pitti & Boboli Gardens", "type": "museum/park", "start_time": "10:00", "price_eur": 18},
    {"name": "Sunrise hike to Piazzale Michelangelo", "type": "outdoor", "start_time": "05:45", "price_eur": 0},
    {"name": "Duomo & Brunelleschi's Dome climb", "type": "landmark", "start_time": "09:30", "price_eur": 30},
    {"name": "Ponte Vecchio evening walk", "type": "outdoor", "start_time": "18:00", "price_eur": 0},
]
RESTAURANTS = [
    {"name": "Trattoria Mario", "cuisine": "Italian", "price": "€€", "open": "12:00-15:00"},
    {"name": "Osteria Santo Spirito", "cuisine": "Italian", "price": "€€", "open": "19:00-23:00"},
    {"name": "Il Latini", "cuisine": "Italian", "price": "€€€", "open": "19:30-22:30"},
    {"name": "Sushi Zen", "cuisine": "Japanese", "price": "€€", "open": "19:00-22:30"},
]

@tool
def search_flights(origin: str, destination: str, date: str = "this weekend") -> str:
    """Search flights. Use origin='HOME' if the user has not said where they fly from."""
    _pause()
    return json.dumps({"origin": origin, "destination": destination, "date": date, "flights": [
        {"flight": "AZ1593", "depart": "06:10", "arrive": "08:05", "price_eur": 120},
        {"flight": "LH1832", "depart": "11:40", "arrive": "13:35", "price_eur": 165},
        {"flight": "BA0610", "depart": "17:25", "arrive": "19:20", "price_eur": 140}]})

@tool
def search_hotels(city: str, max_price_per_night_eur: int = 0) -> str:
    """Search hotels in a city. Pass max_price_per_night_eur to respect the user's budget (0 = no limit)."""
    _pause()
    hotels = [h for h in HOTELS if not max_price_per_night_eur or h["price_per_night_eur"] <= max_price_per_night_eur]
    return json.dumps({"city": city, "hotels": hotels})

@tool
def find_attractions(city: str, interest: str = "") -> str:
    """Find attractions in a city, optionally filtered by an interest such as 'museum'. Results include start times."""
    _pause()
    items = [a for a in ATTRACTIONS if not interest or interest.lower() in a["type"].lower()] or ATTRACTIONS
    return json.dumps({"city": city, "attractions": items})

@tool
def search_restaurants(city: str, cuisine: str = "") -> str:
    """Search restaurants in a city, optionally filtered by cuisine (e.g. 'Italian')."""
    _pause()
    items = [r for r in RESTAURANTS if not cuisine or cuisine.lower() in r["cuisine"].lower()] or RESTAURANTS
    return json.dumps({"city": city, "restaurants": items})

@tool
def get_weather(city: str, days: int = 3) -> str:
    """Get a short weather forecast for a city."""
    _pause()
    return json.dumps({"city": city, "forecast": [
        {"day": "Saturday", "condition": "Sunny", "high_c": 24, "low_c": 14},
        {"day": "Sunday", "condition": "Light rain in the afternoon", "high_c": 21, "low_c": 13}][:max(days, 1)]})

TOOLS = [search_flights, search_hotels, find_attractions, search_restaurants, get_weather]

## 4 · Memory writer + visible planner (both LangChain structured-output calls)

In [ ]:
class Pref(BaseModel):
    category: Literal["likes", "avoids", "budget", "other"]
    value: str

class Extracted(BaseModel):
    preferences: list[Pref] = Field(default_factory=list)
    style_rules: list[str] = Field(default_factory=list)

EXTRACT_PROMPT = """Extract ONLY durable travel preferences or answer-style rules the user explicitly states in this message.
- likes: things they enjoy (e.g. "museums", "Italian food")
- avoids: things they dislike or want to avoid (e.g. "early morning activities")
- budget: their preferred budget / price range (e.g. "100-150 EUR per night")
- style_rules: how they want answers formatted (e.g. "keep answers short")
Do not invent anything. A request like "Plan me a weekend in Florence" contains no preference -> return empty lists."""

extractor = llm.with_structured_output(Extracted)

def extract_and_save(user_id, text):
    try:
        res = extractor.invoke([("system", EXTRACT_PROMPT), ("user", text)])
    except Exception as e:
        print("extract error:", e)
        return []
    saved = []
    for p in (res.preferences if res else []):
        memory.save_preference(user_id, p.category, p.value)
        saved.append(f"{p.category}: {p.value}")
    for r in (res.style_rules if res else []):
        memory.add_rule(user_id, r)
        saved.append(f"rule: {r}")
    return saved

StepKind = Literal["load_preferences", "search_flights", "search_hotels", "find_attractions",
                   "search_restaurants", "get_weather", "final_answer"]

class Plan(BaseModel):
    steps: list[StepKind]

LABELS = {"load_preferences": "Load user preferences", "search_flights": "Search flights",
          "search_hotels": "Search hotels", "find_attractions": "Find attractions",
          "search_restaurants": "Search restaurants", "get_weather": "Check weather",
          "final_answer": "Build final itinerary / reply"}

PLAN_PROMPT = """You are the planner of a travel assistant. Return the ordered steps needed to answer the user's latest message.
Allowed steps: load_preferences, search_flights, search_hotels, find_attractions, search_restaurants, get_weather, final_answer.
- Always start with load_preferences and end with final_answer.
- For trip-planning requests (e.g. "plan a weekend in X") include flights, hotels, attractions, restaurants and weather.
- For a narrower request (e.g. "just find hotels") include only the relevant tool steps.
- For greetings, preference statements or questions about memory, return only [load_preferences, final_answer]."""

planner = llm.with_structured_output(Plan)

def make_plan(user_text):
    try:
        kinds = list(planner.invoke([("system", PLAN_PROMPT), ("user", user_text)]).steps)
    except Exception as e:
        print("planner error, using fallback:", e)
        kinds = ["search_flights", "search_hotels", "find_attractions", "search_restaurants", "get_weather"] \
            if any(w in user_text.lower() for w in ("plan", "trip", "weekend", "itinerary")) else []
    kinds = [k for k in dict.fromkeys(kinds) if k not in ("load_preferences", "final_answer")]
    kinds = ["load_preferences"] + kinds + ["final_answer"]
    return [{"kind": k, "label": LABELS[k], "status": "pending"} for k in kinds]

## 5 · The agent (`create_agent` + SQLite checkpointer) and an event-streaming runner
* **Short-term memory** = LangGraph `SqliteSaver` checkpointer, keyed by `thread_id`.
* **Long-term memory** = `MemoryStore`, keyed by `user_id`, injected into the system prompt on every turn.
* The runner yields UI state (plan, tool log, answer) after every graph step, mapping tool events → plan status.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver

checkpointer = SqliteSaver(sqlite3.connect(CHECKPOINT_DB, check_same_thread=False))

def reconnect():
    """Simulate an application restart: drop every in-memory object and re-open the DB files."""
    global memory, checkpointer
    memory = MemoryStore(MEMORY_DB)
    checkpointer = SqliteSaver(sqlite3.connect(CHECKPOINT_DB, check_same_thread=False))

def _text(content):
    if isinstance(content, str):
        return content
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)

def build_system_prompt(user_id, plan):
    m = memory.get_all(user_id)
    prefs = "\n".join(f"- {c}: {v}" for c, v in m["preferences"]) or "- (nothing saved yet)"
    eps = "\n".join(f"- {d}: {s}" for d, s in m["episodes"]) or "- (none)"
    rules = "\n".join(f"- {r}" for r in m["rules"])
    steps = "\n".join(f"{i+1}. {s['label']}" for i, s in enumerate(plan))
    return f"""You are a friendly travel-planning assistant with memory. All tools return MOCK data.

## Long-term memory about this user (semantic)
{prefs}

## Past interactions (episodic)
{eps}

## Behaviour rules (procedural)
{rules}

## Plan for this request (already shown to the user)
{steps}

Follow the plan: call the planned tools (you may call independent tools together), then write the final answer.
Apply the saved preferences automatically (e.g. use the budget for hotel searches, avoid activities that start early
if the user avoids early mornings, favour liked interests/cuisines) and tell the user which ones you applied.
If the user only states a preference, acknowledge it briefly. Do not claim to have booked anything."""

def run_agent(user_id, thread_id, user_text):
    """Generator: yields a state dict {plan, log, answer} after every step."""
    state = {"plan": [], "log": [], "answer": None}
    try:
        saved = extract_and_save(user_id, user_text)                      # write long-term memory
        if saved:
            state["log"].append({"name": "save_memory", "args": {"saved": saved}, "result": "stored in SQLite", "id": "mem"})
        plan = state["plan"] = make_plan(user_text)                       # visible plan BEFORE any tool call
        yield state

        def mark(kind, status):
            for s in plan:
                if s["kind"] == kind:
                    s["status"] = status
                    return
            plan.insert(len(plan) - 1, {"kind": kind, "label": kind, "status": status})   # unplanned tool

        def advance_final():
            if all(s["status"] == "done" for s in plan[:-1]) and plan[-1]["status"] == "pending":
                plan[-1]["status"] = "running"

        mark("load_preferences", "running"); yield state
        loaded = memory.get_all(user_id)
        state["log"].append({"name": "load_user_preferences", "args": {"user_id": user_id},
                             "result": json.dumps({"preferences": loaded["preferences"]}, ensure_ascii=False), "id": "load"})
        mark("load_preferences", "done"); advance_final(); yield state

        agent = create_agent(llm, TOOLS, system_prompt=build_system_prompt(user_id, plan), checkpointer=checkpointer)
        config = {"configurable": {"thread_id": thread_id}}
        final = ""
        for chunk in agent.stream({"messages": [HumanMessage(user_text)]}, config, stream_mode="updates"):
            for _node, update in chunk.items():
                if not isinstance(update, dict):
                    continue
                for msg in update.get("messages") or []:
                    if isinstance(msg, AIMessage):
                        if msg.tool_calls:
                            for tc in msg.tool_calls:
                                mark(tc["name"], "running")
                                state["log"].append({"name": tc["name"], "args": tc["args"], "result": None, "id": tc["id"]})
                        else:
                            final = _text(msg.content)
                    elif isinstance(msg, ToolMessage):
                        mark(msg.name, "done")
                        for e in state["log"]:
                            if e["id"] == msg.tool_call_id:
                                e["result"] = _text(msg.content)
                        advance_final()
            yield state

        plan[-1]["status"] = "done"
        state["answer"] = final or "(no answer)"
        used = [e["name"] for e in state["log"] if e["id"] not in ("mem", "load")]
        if used:                                                           # write episodic memory
            memory.add_episode(user_id, summarize_episode(user_text, final, used))
        yield state
    except Exception as e:
        state["answer"] = f"⚠️ Error: {type(e).__name__}: {e}"
        yield state

def summarize_episode(user_text, answer, tools):
    try:
        r = llm.invoke(f"Summarize in ONE sentence (max 35 words) what trip was planned and the key choices made, "
                       f"for future reference.\nUser request: {user_text}\nAssistant answer: {answer[:1500]}")
        text = _text(r.content).strip()
        if text:
            return text
    except Exception:
        pass
    return f"User asked: '{user_text[:80]}' (tools used: {', '.join(tools)})"

## 6 · Gradio UI — chat, plan + progress, tool calls/results, saved memories

In [ ]:
import gradio as gr

ICON = {"pending": "☐", "running": "⟳", "done": "☑"}

def render_plan(plan):
    if not plan:
        return "_No plan yet — send a message._"
    done = sum(s["status"] == "done" for s in plan)
    bar = "▰" * done + "▱" * (len(plan) - done)
    lines = [f"- {ICON[s['status']]} {s['label']}" + ("  *(running…)*" if s["status"] == "running" else "") for s in plan]
    return f"**Progress {bar} {done}/{len(plan)}**\n\n" + "\n".join(lines) + "\n\n<sub>☑ Completed · ⟳ Running · ☐ Pending</sub>"

def render_tools(log):
    if not log:
        return "_No tool calls yet._"
    out = []
    for e in log:
        res = e["result"] if e["result"] is not None else "⟳ running…"
        out.append(f"**🔧 {e['name']}**  `{json.dumps(e['args'], ensure_ascii=False)}`\n```json\n{res[:600]}\n```")
    return "\n\n".join(out)

def short_term_messages(thread_id):
    try:
        t = checkpointer.get_tuple({"configurable": {"thread_id": thread_id}})
        return t.checkpoint["channel_values"].get("messages", []) if t else []
    except Exception:
        return []

def render_memory(user_id, thread_id):
    m = memory.get_all(user_id)
    prefs = "\n".join(f"- **{c}**: {v}" for c, v in m["preferences"]) or "- _nothing saved yet_"
    eps = "\n".join(f"- `{d}` {s}" for d, s in m["episodes"]) or "- _none yet_"
    rules = "\n".join(f"- {r}" for r in m["rules"])
    n = len(short_term_messages(thread_id))
    return (f"#### 🗄️ Long-term · persistent (SQLite) · user `{user_id}`\n"
            f"**Semantic — preferences**\n{prefs}\n\n**Episodic — past sessions**\n{eps}\n\n**Procedural — rules**\n{rules}\n\n"
            f"#### 💬 Short-term · thread `{thread_id}`\n- {n} messages in the checkpointer")

def new_thread():
    return f"thread-{uuid.uuid4().hex[:6]}"

def load_history(thread_id):
    hist = []
    for m in short_term_messages(thread_id):
        if isinstance(m, HumanMessage):
            hist.append({"role": "user", "content": _text(m.content)})
        elif isinstance(m, AIMessage) and not m.tool_calls and _text(m.content).strip():
            hist.append({"role": "assistant", "content": _text(m.content)})
    return hist

def on_send(message, history, user_id, thread_id):
    message = (message or "").strip()
    if not message:
        yield history, "", gr.skip(), gr.skip(), gr.skip()
        return
    history = (history or []) + [{"role": "user", "content": message}]
    wait = history + [{"role": "assistant", "content": "⏳ Working on it…"}]
    yield wait, "", "_planning…_", "", render_memory(user_id, thread_id)
    for st in run_agent(user_id, thread_id, message):
        shown = history + [{"role": "assistant", "content": st["answer"]}] if st["answer"] else wait
        yield shown, "", render_plan(st["plan"]), render_tools(st["log"]), render_memory(user_id, thread_id)

try:
    demo.close()
except NameError:
    pass

with gr.Blocks(title="Travel Agent with Memory", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧭 Travel Agent with Memory\nShort-term memory = *thread*, long-term memory = *user*. "
                "Tell it your preferences, press **New conversation**, then ask for a trip.")
    with gr.Row():
        user_box = gr.Textbox(value="demo_user", label="User ID (long-term memory key)", scale=2)
        thread_box = gr.Textbox(value=new_thread(), label="Thread ID (short-term memory key)", scale=2)
        new_btn = gr.Button("➕ New conversation", scale=1)
        load_btn = gr.Button("↻ Load thread history", scale=1)
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages", height=480, label="Chat")
            msg = gr.Textbox(placeholder="e.g. I like museums and Italian food, and I don't like early morning activities.",
                             show_label=False)
            gr.Examples(["I like museums and Italian food, and I don't like early morning activities.",
                         "My budget is around 100-150 EUR per night for hotels.",
                         "Plan me a weekend in Florence.",
                         "What do you remember about my preferences?"], inputs=msg)
        with gr.Column(scale=2):
            with gr.Accordion("📋 Current plan & progress", open=True):
                plan_md = gr.Markdown(render_plan([]))
            with gr.Accordion("🔧 Tool calls & results", open=True):
                tools_md = gr.Markdown(render_tools([]))
            with gr.Accordion("🧠 Saved memories", open=True):
                mem_md = gr.Markdown(render_memory("demo_user", thread_box.value))

    msg.submit(on_send, [msg, chatbot, user_box, thread_box], [chatbot, msg, plan_md, tools_md, mem_md])
    new_btn.click(lambda u: (new_thread(), [], render_plan([]), render_tools([]), render_memory(u, "-")),
                  [user_box], [thread_box, chatbot, plan_md, tools_md, mem_md])
    load_btn.click(lambda u, t: (load_history(t), render_memory(u, t)), [user_box, thread_box], [chatbot, mem_md])
    user_box.change(render_memory, [user_box, thread_box], mem_md)

demo.queue().launch(share=True, debug=False)   # Kaggle: open the public gradio.live link printed below

## 7 · Scripted demo — proves long-term memory survives an application restart
Conversation 1 states preferences → we **simulate a restart** (new objects, files re-opened) → Conversation 2 (a brand-new thread, so
no chat history exists) asks for a trip, and the agent must use the preferences from SQLite.
For the *real* thing: after Conversation 1, restart the kernel, re-run the cells, and only send Conversation 2 in the UI.

In [ ]:
USER = "demo_user"

def chat_once(thread_id, text):
    print(f"\n👤 {text}")
    last_plan = None
    for st in run_agent(USER, thread_id, text):
        snap = " | ".join(f"{ICON[s['status']]} {s['label']}" for s in st["plan"])
        if snap != last_plan:
            print("   plan:", snap)
            last_plan = snap
    print("🤖", st["answer"])

# memory.reset(USER)   # uncomment to start from a clean slate
chat_once(new_thread(), "I like museums and Italian food, and I don't like early morning activities. My hotel budget is about 100-150 EUR per night.")
print("\n--- 🔄 simulated application restart ---")
reconnect()
print(render_memory(USER, "-"))
chat_once(new_thread(), "Plan me a weekend in Florence.")

## 8 · How legacy LangChain memory relates to LangGraph memory
Legacy classes (`ConversationBufferMemory`, `...WindowMemory`, `...SummaryMemory`, retriever memory) lived **inside chains** and
only managed *one thing*: the chat transcript that gets pasted into the prompt. They were deprecated in LangChain 0.3.1 and now live in `langchain-classic`.

LangGraph/LangChain v1 splits that single idea in two and makes both **explicit, persistent, and keyed**:
| Legacy | Modern equivalent |
|---|---|
| Buffer memory (whole transcript) | **Short-term memory**: graph state saved by a *checkpointer* per `thread_id` |
| Window / token-buffer memory | Trim messages (`trim_messages`) or middleware that limits what the model sees |
| Summary / summary-buffer memory | `SummarizationMiddleware` (summarise old messages automatically) |
| Vector-store retriever memory | **Long-term memory**: a `Store` (namespace + key → JSON) with semantic search |
| *(no equivalent)* | Semantic / episodic / procedural memory patterns, shared across threads and restarts |

In [ ]:
# Optional: run a legacy memory class next to its modern replacement
try:
    import warnings; warnings.filterwarnings("ignore")
    from langchain_classic.memory import ConversationBufferWindowMemory
    legacy = ConversationBufferWindowMemory(k=2)
    for i, (q, a) in enumerate([("Hi, I like museums", "Noted!"), ("Plan Rome", "Sure"), ("Add hotels", "Done")]):
        legacy.save_context({"input": q}, {"output": a})
    print("LEGACY window memory (k=2) only 'remembers':\n", legacy.load_memory_variables({})["history"])
except Exception as e:
    print("langchain-classic not available:", e)
print("\nMODERN: short-term messages for the last UI thread are persisted in", CHECKPOINT_DB, "and preferences in", MEMORY_DB)